# CT107-3-3 Text Analytics and Sentiment Analysis
## Part A — Q5 (Individual Component): Alternative Tokenization
### Student 4 — Alternative Approach: **BERT WordPiece Tokenizer** (HuggingFace Transformers)
**Dataset:** `Data_1.txt`  
*(10 marks)*

### Setup — Import Libraries and Load Data

In [1]:
import re
import nltk
from nltk.tokenize import word_tokenize
from transformers import BertTokenizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

DATA_PATH = r"D:\TXSA\Part A Dataset\Part A Dataset\Data_1.txt"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = f.read().strip()

text = re.sub(r'^\d+\t', '', raw, flags=re.MULTILINE)

print("Text corpus:")
print(text)

Text corpus:
Classification is the task of choosing the correct class label for a given input. In basic
classification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.


---
## Q5(a) — Implement Tokenization using BERT WordPiece Tokenizer
*(3 marks)*

**BERT (Bidirectional Encoder Representations from Transformers)** uses a **WordPiece tokenizer** — a subword tokenization algorithm. Unlike word-level tokenizers that split on whitespace and punctuation, WordPiece learns a vocabulary of ~30,000 **subword units** from a large corpus (Wikipedia + BooksCorpus) that balances:
- **Frequent whole words** as single tokens (e.g., `classification`)
- **Rare or unknown words** split into known subword pieces (e.g., `multiclass` → `multi` + `##class`)

The `##` prefix indicates a subword continuation (the piece attaches to the previous token). This eliminates out-of-vocabulary (OOV) words entirely.

In [2]:
# Load pre-trained BERT base uncased tokenizer
# (downloads ~200KB vocab file on first run)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Basic tokenization (WordPiece subwords)
tokens_bert = tokenizer.tokenize(text)

print(f"[BERT WordPiece Tokenizer]  →  {len(tokens_bert)} tokens")
print(tokens_bert)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[BERT WordPiece Tokenizer]  →  97 tokens
['classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'in', 'basic', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', '.', 'the', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants', '.', 'for', 'example', ',', 'in', 'multi', '##class', 'classification', ',', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', ';', 'in', 'open', '-', 'class', 'classification', ',', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', ';', 'and', 'in', 'sequence', 'classification', ',', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'classified', '.']


In [3]:
# Full encoding with special tokens [CLS] and [SEP]
# BERT requires these boundary markers for its attention mechanism
encoded = tokenizer(text, return_tensors=None)
input_ids = encoded['input_ids']
decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)

print(f"With special tokens [CLS] and [SEP]:")
print(f"  Token IDs : {input_ids}")
print(f"  Tokens    : {decoded_tokens}")
print(f"  Count     : {len(decoded_tokens)} (including [CLS] and [SEP])")

With special tokens [CLS] and [SEP]:
  Token IDs : [101, 5579, 2003, 1996, 4708, 1997, 10549, 1996, 6149, 2465, 3830, 2005, 1037, 2445, 7953, 1012, 1999, 3937, 5579, 8518, 1010, 2169, 7953, 2003, 2641, 1999, 12477, 2013, 2035, 2060, 20407, 1010, 1998, 1996, 2275, 1997, 10873, 2003, 4225, 1999, 5083, 1012, 1996, 3937, 5579, 4708, 2038, 1037, 2193, 1997, 5875, 10176, 1012, 2005, 2742, 1010, 1999, 4800, 26266, 5579, 1010, 2169, 6013, 2089, 2022, 4137, 3674, 10873, 1025, 1999, 2330, 1011, 2465, 5579, 1010, 1996, 2275, 1997, 10873, 2003, 2025, 4225, 1999, 5083, 1025, 1998, 1999, 5537, 5579, 1010, 1037, 2862, 1997, 20407, 2024, 10776, 6219, 1012, 102]
  Tokens    : ['[CLS]', 'classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'in', 'basic', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined'

In [4]:
# Highlight which tokens are subword pieces (starting with ##)
whole_words  = [t for t in tokens_bert if not t.startswith('##')]
subword_cont = [t for t in tokens_bert if t.startswith('##')]

print(f"Whole-word tokens  ({len(whole_words)}) : {whole_words}")
print(f"Subword pieces     ({len(subword_cont)}) : {subword_cont}")

# Reconstruct original words from subword pieces
print("\nWord reconstruction from subword pieces:")
words_reconstructed = []
current_word = ""
for token in tokens_bert:
    if token.startswith("##"):
        current_word += token[2:]
    else:
        if current_word:
            words_reconstructed.append(current_word)
        current_word = token
if current_word:
    words_reconstructed.append(current_word)

print(words_reconstructed)

Whole-word tokens  (96) : ['classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'in', 'basic', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', '.', 'the', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants', '.', 'for', 'example', ',', 'in', 'multi', 'classification', ',', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', ';', 'in', 'open', '-', 'class', 'classification', ',', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', ';', 'and', 'in', 'sequence', 'classification', ',', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'classified', '.']
Subword pieces     (1) : ['##class']

Word reconstruction from subword pieces:
['classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correc

In [5]:
# Demonstrate OOV handling: BERT never produces [UNK] for English text
rare_words = ["multiclassification", "tokenizationability", "supercalifragilistic", "NLP2024"]

print("OOV Handling Demo — Rare / unseen words:")
print(f"{'Input word':<30} {'NLTK output':<25} {'BERT WordPiece output'}")
print("-" * 80)
for word in rare_words:
    nltk_out = word_tokenize(word)
    bert_out = tokenizer.tokenize(word)
    print(f"{word:<30} {str(nltk_out):<25} {bert_out}")

OOV Handling Demo — Rare / unseen words:
Input word                     NLTK output               BERT WordPiece output
--------------------------------------------------------------------------------
multiclassification            ['multiclassification']   ['multi', '##class', '##ification']
tokenizationability            ['tokenizationability']   ['token', '##ization', '##ability']
supercalifragilistic           ['supercalifragilistic']  ['super', '##cal', '##if', '##rag', '##ilis', '##tic']
NLP2024                        ['NLP2024']               ['nl', '##p', '##20', '##24']


---
## Q5(b) — Compare with Group's Approach (NLTK word_tokenize)
*(2 marks)*

In [6]:
tokens_nltk = word_tokenize(text)

print(f"{'Method':<35} {'Token Count':<15}")
print("-" * 50)
print(f"{'NLTK word_tokenize (group)':<35} {len(tokens_nltk)}")
print(f"{'BERT WordPiece (mine)':<35} {len(tokens_bert)}")

print(f"\nNLTK tokens : {tokens_nltk}")
print(f"BERT tokens : {tokens_bert}")

Method                              Token Count    
--------------------------------------------------
NLTK word_tokenize (group)          94
BERT WordPiece (mine)               97

NLTK tokens : ['Classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'In', 'basic', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', '.', 'The', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants', '.', 'For', 'example', ',', 'in', 'multiclass', 'classification', ',', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', ';', 'in', 'open-class', 'classification', ',', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', ';', 'and', 'in', 'sequence', 'classification', ',', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'cl

In [7]:
features = [
    ("Tokenization level",                "Word-level",                "Subword (WordPiece)"),
    ("Vocabulary size",                   "Unlimited",                 "~30,000 fixed subwords"),
    ("Handles OOV words",                 "Keeps as-is (no UNK)",      "Splits into known subwords"),
    ("Token count (Data_1)",              str(len(tokens_nltk)),       str(len(tokens_bert))),
    ("Lowercases by default",             "No",                        "Yes (uncased model)"),
    ("Special tokens",                    "None",                      "[CLS], [SEP], [PAD]"),
    ("Designed for Transformers",         "No",                        "Yes"),
    ("Requires pretrained model",         "No",                        "Yes (~200KB vocab)"),
    ("Output type",                       "List of strings",           "List of subword strings"),
    ("Suitable for BERT/GPT",             "No (incompatible)",         "Yes (required)"),
    ("Interpretability",                  "High (whole words)",        "Lower (subword pieces)"),
]

print(f"{'Feature':<35} {'NLTK word_tokenize':<30} {'BERT WordPiece'}")
print("-" * 90)
for feat, nltk_val, bert_val in features:
    print(f"{feat:<35} {nltk_val:<30} {bert_val}")

Feature                             NLTK word_tokenize             BERT WordPiece
------------------------------------------------------------------------------------------
Tokenization level                  Word-level                     Subword (WordPiece)
Vocabulary size                     Unlimited                      ~30,000 fixed subwords
Handles OOV words                   Keeps as-is (no UNK)           Splits into known subwords
Token count (Data_1)                94                             97
Lowercases by default               No                             Yes (uncased model)
Special tokens                      None                           [CLS], [SEP], [PAD]
Designed for Transformers           No                             Yes
Requires pretrained model           No                             Yes (~200KB vocab)
Output type                         List of strings                List of subword strings
Suitable for BERT/GPT               No (incompatible)           

---
## Q5(c) — Why BERT WordPiece is Different / Better / Worse
*(5 marks)*

### How BERT WordPiece Tokenization Works
WordPiece tokenization is a **data-driven subword algorithm** trained on a large corpus. The vocabulary is built by:
1. Starting with all individual characters as the base vocabulary
2. Iteratively merging the pair of symbols that maximises the **likelihood** of the training data when added to the vocabulary
3. Stopping when the vocabulary reaches the target size (~30,000)

At inference time, an input word is greedily split into the **longest matching subword sequences** from the learned vocabulary, using `##` to mark continuation pieces.

---

### Where BERT WordPiece is **Better**

| Scenario | Why BERT WordPiece wins |
|----------|-------------------------|
| **Zero OOV words** | WordPiece can represent **any** word by decomposing it to characters if needed. NLTK passes unknown words through unchanged — which is fine for known text but problematic for domain-specific neologisms (e.g., `COVID-19`, `multiclassification`). |
| **Transformer compatibility** | BERT, GPT, RoBERTa and all modern transformer models **require** their own tokenizer. Using NLTK tokens with BERT will produce incorrect results because the model's embeddings are indexed by WordPiece IDs, not word strings. |
| **Fixed vocabulary** | A fixed 30K-token vocabulary makes memory requirements predictable and enables efficient batch processing across a corpus of any size. |
| **Morphological sharing** | Related words share subword representations: `classification`, `classify`, `classified` all share the subword `classif##`, so the model inherently understands their relationship even without stemming. |

### Where BERT WordPiece is **Worse** / Different

| Scenario | Why NLTK word_tokenize is better |
|----------|----------------------------------|
| **Human readability** | Subword tokens like `##ifi`, `##cat`, `##ion` are not meaningful to humans. NLTK tokens are whole words, making outputs easy to interpret and debug. |
| **Token count inflation** | WordPiece produces more tokens than NLTK for rare or long words, which increases sequence length and computational cost in downstream models. |
| **Simplicity** | BERT's tokenizer requires downloading a pre-trained vocab file and using the HuggingFace `transformers` library. NLTK requires no external dependencies for tokenization. |
| **Classic NLP tasks** | For bag-of-words models, TF-IDF, n-gram language models, and traditional ML classifiers, NLTK word tokens are the standard input. BERT subwords are meaningless in these contexts. |
| **Case sensitivity** | `bert-base-uncased` lowercases everything, losing information about proper nouns (`Danielle` → `danielle`). NLTK preserves case by default. |

### Conclusion
BERT WordPiece tokenization is **fundamentally different** from NLTK `word_tokenize` — it operates at the **subword level** rather than the word level. It is **significantly better** when the goal is to build or fine-tune **transformer-based models** (BERT, RoBERTa, DistilBERT) for tasks such as text classification, named entity recognition, or question answering, because the tokenization is inseparable from the model's vocabulary and embeddings.

For the tasks in this assignment — basic text processing, stemming, POS tagging, and n-gram language models — BERT tokenization is **worse** (or simply incompatible), because those tasks expect **whole-word tokens** as input. The BERT tokenizer shines in the context of **deep learning text classification** (Part B of this assignment), where it would be the natural choice for a transformer-based classifier.